In [ ]:
import os
os.environ['ATTN_BACKEND'] = 'flash-attn'   # Can be 'flash-attn' or 'xformers', default is 'flash-attn'
os.environ['SPCONV_ALGO'] = 'auto'        # Can be 'native' or 'auto', default is 'auto'.
                                            # 'auto' is faster but will do benchmarking at the beginning.
                                            # Recommended to set to 'native' if run only once.
os.environ['OMP_NUM_THREADS'] = '4'        # Set the number of threads for OpenMP. Default is 4.
os.environ['TOKENIZERS_PARALLELISM'] = 'false' 
os.environ['USE_FLUX_DEV'] = 'true' 
os.environ['ENABLE_FLUX_CPU_OFFLOAD'] = 'true'  

from matplotlib import pyplot as plt
import rembg
import torch
from IPython.display import Video

# Import modular components
from modules.model_manager import get_model_manager
from modules.content_moderation import get_content_moderator  
from modules.generation_pipeline import get_generation_pipeline

# CONFIGURATION SETUP 
print("🔧 Configuring TRELLIS pipeline...")
device_config = {
    "flux": "cuda:0",     # FLUX with CPU offloading for max efficiency  
    "trellis": "cuda:0"   # TRELLIS on dedicated GPU (if available)
}

# Auto-detect best configuration based on available hardware
num_gpus = torch.cuda.device_count()

if num_gpus >= 2 and device_config["flux"] != device_config["trellis"]:
    # Multi-GPU mode: TRELLIS CPU offloading will be automatically disabled
    model_manager = get_model_manager(device_config, enable_trellis_cpu_offload=True)  # Will be auto-disabled
else:
    # Single-GPU mode: Enable TRELLIS CPU offloading for memory efficiency
    device_config = {"flux": "cuda:0", "trellis": "cuda:0"} 
    model_manager = get_model_manager(device_config, enable_trellis_cpu_offload=True)

content_moderator = get_content_moderator()
generation_pipeline = get_generation_pipeline()

# Load all models
flux_pipe, trellis_pipeline, reward_model = model_manager.load_all_models()

# Configure generation pipeline with models
generation_pipeline.set_models(
    flux_pipeline=flux_pipe,
    trellis_pipeline=trellis_pipeline,
    reward_model=reward_model,
    content_moderator=content_moderator
)

# Get generation config
config = model_manager.get_generation_config()
guidance_scale = config["guidance_scale"]
num_inference_steps = config["num_inference_steps"]

print("\n✅ All modules initialized and models loaded")
print(f"📊 Configuration: guidance_scale={guidance_scale}, num_inference_steps={num_inference_steps}")

# Check GPU memory usage
print(f"GPU Memory Usage:")
for i in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(i) / 1024**3
    reserved = torch.cuda.memory_reserved(i) / 1024**3
    total = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"  GPU {i}: {allocated:.1f}GB / {total:.1f}GB allocated")

In [ ]:
prompt = "a pretzel-copter, a helicopter where the cockpit is in a pretzel shape and the whole helicopter looks like a pretzel"
prompt_suffix = " Render of high quality 3D model on neutral background. Solid, contiguous mesh, optimized for 3D printing."
prompt = prompt + prompt_suffix

batch_size = 4

# Check text safety first
is_safe, scores = content_moderator.check_text_safety(prompt)
if not is_safe:
    print(f"⚠️ Prompt flagged by content moderation: {scores}")
    print("Please use a different prompt.")
else:
    print("✅ Prompt passed content moderation")
    
    # Use generation pipeline to create images
    print("🎨 Generating images...")
    filtered_images, images = generation_pipeline.generate_images_return_raw(
        prompt, 
        num_images=batch_size,
        base_seed=13,  # Fixed seed for reproducibility
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps
    )
    
    print(f"✅ Generated {len(filtered_images)} images")
    
    # Display images with background removal
    plt.figure(figsize=(20, 15))
    for i in range(len(filtered_images)):
        plt.subplot(1, len(filtered_images), i + 1)
        image = filtered_images[i]
        output = rembg.remove(image)
        plt.imshow(output)
        plt.axis('off')
    plt.show()

In [ ]:
best_image_idx = 1
selected_image = filtered_images[best_image_idx]
selected_image

In [ ]:
# Use generation pipeline for 3D model creation with CPU offloading and simple GLB export
video_path, glb_path = generation_pipeline.generate_3d_model(
    selected_image,
    sample_video=True,
    texture_size=512,
    base_seed=3,  # Fixed seed for reproducibility
    use_simple_glb=False  # Use memory-efficient GLB export to avoid 66TB bug
)

In [ ]:
# if len(video_path) > 0:
Video(video_path, embed=True, width=800)

In [ ]:
# get peak gpu memory usage
for i in range(torch.cuda.device_count()):
    peak = torch.cuda.max_memory_allocated(i) / 1024**3
    print(f"🔝 Peak GPU {i} Memory Usage: {peak:.1f}GB")